In [ ]:
import torch
from jupyterlab.semver import valid
from torch import nn

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, num_dims, epsilon=0e-5):
        super().__init__()
        self.alpha = torch.ones(num_dims, requires_grad=True).reshape(1, 1, num_dims)
        self.beta = torch.zeros(num_dims, requires_grad=True).reshape(1, 1, num_dims)
        self.epsilon = epsilon

    def forward(self, x):
        x_mean = x.mean(-1, keepdim=True)  #shape=(batch_size, seq, 1)
        #虽然可以直接求标准差...
        x_var = x.var(-1, keepdim=True)
        return (x - x_mean) / torch.sqrt(x_var + self.epsilon) * self.alpha + self.beta

In [ ]:
def softmax(x, dim=-1):
    x_exp = x.exp()
    
    x_exp_sum = x_exp.sum(dim=dim, keepdim=True)
    return x_exp / x_exp_sum

def masked_softmax(x, valid_lens):
    if valid_lens is None:
        # 推理阶段的解码器输入 
        return softmax(x)
    # x: 3D tensor, valid_lens: 1D or 2D tensor
    """
    x.shape == (b, seq, seq)
    valid_lens 在训练阶段和推理阶段，编码器端为(b*head)
    在训练阶段，解码器端为(b*head, arange(max_len+1)*head)
    在推理阶段，解码器端为None，但是为什么？
    真难坚持啊
    """
    
    shape = x.shape
    valid_len_1D = None
    if valid_lens.dim() == 1:
        valid_len_1D = valid_lens.repeat_interleave(repeats=shape[1])
    elif valid_lens.dim() == 2:
        valid_len_1D = valid_lens.reshape(-1) #reshape后形状应为(batch_size * seq)
    else:
        pass # 抛出异常
    masked_scores = mask_seqs(x.reshape(-1, shape[1]), valid_len_1D, value=-1e6).reshape(shape) #masked_scores = [b, seq, seq]
    return softmax(masked_scores)

def mask_seqs(x, valid_lens, value=0):
    """
    :param x: x.shape=[b, seq, ...] 只会沿着seq其代表的张量进行覆盖
    :param valid_lens: 限制为1D，因为x的输入就是(b,seq) 2D,不能比x的维度大，且该函数本来就是用来遮盖自然语言中的无效词元
    :param value:
    :return: 
    """
    shape = x.shape
    cond = torch.arange(shape[1], dtype=torch.float32).unsqueeze(0).expand(shape[0], -1) < valid_lens.unsqueeze(-1) #在最后一维后面再加一维，维数为1然后进行广播
    x[~cond] = value
    return x
    

In [ ]:
def dot_attention(query, key, value, valid_lens):
    num_dims = query.shape[-1]
    # key与value形状一样
    # q = (b, seq1, dim)
    # kv = (b, seq2, dim)
    # 犯了个错误，q与k的矩阵是注意力分数，首先要除以维度的平方，然后要进行masked-softmax将无关kv无效化
    scores = query.bmm(key.transpose(1,2)) / torch.sqrt(num_dims)
    attention_weights = masked_softmax(scores, valid_lens)
    return torch.bmm(attention_weights, value)

In [ ]:
class AttentionLayer(nn.Module):
    def __init__(self, num_dims, num_heads, bias=False):
        super().__init__()
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.num_head_dims = num_dims/num_heads
        self.W_q = nn.LazyLinear(num_dims, bias=bias)
        self.W_k = nn.LazyLinear(num_dims, bias=bias)
        self.W_v = nn.LazyLinear(num_dims, bias=bias)
        self.W_o = nn.LazyLinear(num_dims, bias=bias)
        self.attention = dot_attention
        
    # 相信自己，坚持30分钟
    def forward(self, query, key, value, valid_lens):
        queries = self.transform_qkv(self.W_q(query))
        keys = self.transform_qkv(self.W_k(key))
        values = self.transform_qkv(self.W_v(value))
        return self.W_o(self.transform_qkv_back(self.attention(queries, keys, values, valid_lens)))
        
    def transform_qkv(self, x):
        batch_size = x.shape[0]
        return x.reshape(batch_size, -1, self.num_heads, self.num_head_dims).transpose(1, 2).reshape(batch_size * self.num_heads, -1, self.num_head_dims)
    
    # 注解函数形参类型和返回值类型，在开发时可以用，但是运行时并不会实际检查到底符不符合类型
    def transform_qkv_back(self, x:torch.Tensor):
        # x.shape=[b*head, seq, dim]
        shape = x.shape
        return x.reshape(-1, self.num_heads, shape[1], shape[2]).transpose(1, 2).reshape(-1, shape[1], shape[2])
        
   

In [ ]:
class PositionWiseFFN(nn.Module):
    def __init__(self, num_dims, bias=False):
        super().__init__()
        self.layer1 = nn.LazyLinear(num_dims, bias)
        self.layer2 = nn.LazyLinear(num_dims, bias)
        
    def forward(self, x):
        return self.layer2(nn.functional.relu(self.layer1(x)))

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, num_dims, num_heads, bias=False, **kwargs):
        """
        @Author: XingJiliang
        :param num_heads: 
        :param num_dims: 在这里等于词嵌入的维度，设计为可被num_heads整除
        """
        super().__init__()
        self.attention = AttentionLayer(num_dims, bias=bias)
        self.ln1 = LayerNorm(num_dims)
        self.ffn = PositionWiseFFN(num_dims, bias=bias)
        self.ln2 = LayerNorm(num_dims)
        
    def forward(self, x, valid_lens):
        y = self.attention(x, x, x, valid_lens)
        o = self.ln1(x + y)
        return self.ln2(o + self.ffn(o))

In [ ]:
class Encoder(nn.Module):
    def __init__(self, num_layers, num_dims, num_heads, **kwargs):
        super().__init__()
        self.encoder_layers = nn.Sequential()
        for i in range(num_layers):
            self.encoder_layers.add_module("encoder_layer" + i, EncoderLayer(num_dims, num_heads))

    def forward(self, x, valid_lens):
        return self.encoder_layers(x, valid_lens)

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, num_hidden, num_heads, **kwargs):
        super().__init__()
        self.num_hidden = num_hidden
        self.num_heads = num_heads
        self.masked_attention = AttentionLayer(num_hidden, num_heads)
        self.ln1 = LayerNorm(num_hidden)
        self.cross_attention = AttentionLayer(num_hidden, num_heads)
        self.ln2 = LayerNorm(num_hidden)
        self.ffn = PositionWiseFFN(num_hidden)
        self.ln3 = LayerNorm(num_hidden)
        self.W_q1 = nn.LazyLinear(num_hidden, bias=kwargs['bias'])
        self.W_k1 = nn.LazyLinear(num_hidden, bias=kwargs['bias'])
        self.W_v1 = nn.LazyLinear(num_hidden, bias=kwargs['bias'])

    def forward(self, x, valid_lens, enc_outputs):
        shape = x.shape
        if valid_lens is not None:
            # 在训练阶段，使用teacher-forcing方式训练，传进来的是目标真实序列及其真实长度
            torch.arange(1, shape[0] + 1, dtype=torch.float32).repeat_interleave(shape[0])
        else:
            pass
        y1 = self.masked_attention(x, x, x, valid_lens)
        o1 = self.ln1(x + y1)
        #y2 = self.cross_attention(x, )

In [ ]:
class Decoder(nn.Module):
    def __init__(self, num_layers, num_hidden, num_heads, **kwargs):
        super().__init__()
        self.decoder_layers = nn.Sequential()
        for i in range(num_layers):
            self.decoder_layers.add_module("decoder_layer" + i, DecoderLayer(num_hidden, num_heads))
    
    def forward(self, x, valid_lens):
        self.decoder_layers(x, valid_lens)
